# School finance

Finance summaries use only valid reporting records. NULL records remain missing and are never converted to zero.

In [1]:
# ruff: noqa: E402
import os
import sys
from pathlib import Path

import pandas as pd

root = Path(os.environ.get("APEMAP_PROJECT_ROOT", Path.cwd())).resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from apemap.analysis import compute_funding_summary
from apemap.db import get_connection

parliament = int(os.environ.get("APEMAP_PARLIAMENT", "48"))
db_path = Path(
    os.environ.get("APEMAP_DB_PATH", root / "data" / "aped.duckdb")
).resolve()
conn = get_connection(db_path, read_only=True)
finance = compute_funding_summary(conn, parliament)
finance["reporting_year"]

2021

In [2]:
rows = []
for sector, metrics in finance["by_sector"].items():
    rows.append(
        {
            "sector": sector,
            "gross_mean": metrics["gross_income"]["mean"],
            "gross_median": metrics["gross_income"]["median"],
            "gross_n": metrics["gross_income"]["n"],
            "gross_missing": metrics["gross_income"]["missing"],
            "net_mean": metrics["net_recurrent_income"]["mean"],
            "net_n": metrics["net_recurrent_income"]["n"],
            "net_missing": metrics["net_recurrent_income"]["missing"],
        }
    )
pd.DataFrame(rows)

,sector,gross_mean,gross_median,gross_n,gross_missing,net_mean,net_n,net_missing
0,Government,17569.0,17120.0,53,49,17566.0,53,49
1,Catholic,22912.0,21839.0,19,42,19426.0,19,42
2,Independent,32190.0,31604.0,29,15,29109.0,29,15
3,Other,NaN,NaN,0,85,NaN,0,85


In [3]:
conn.close()